## 读取数据

In [1]:
import os
from skimage.io import imread
import numpy as np
from skimage.transform import resize

def read_UC(path):
    # 读取文件夹内所有子文件夹的名称
    subfolders = [name for name in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, name))]

    # 初始化变量
    X = []
    Y = []

    # 遍历每个子文件夹
    for index, subfolder in enumerate(subfolders):
        # 获取子文件夹的路径和类别编号
        subfolder_path = os.path.join(folder_path, subfolder)
        label = index
        
        # 遍历子文件夹内的所有tif图像
        for filename in os.listdir(subfolder_path):
            if filename.endswith('.tif'):
                # 读取图像数据
                image_path = os.path.join(subfolder_path, filename)
                image = imread(image_path)
                image=resize(image,(64,64,3))
                image = np.transpose(image,(2,0,1)) 
                # 将图像数据添加到X变量中
                X.append(image)
                
                # 将类别编号添加到Y变量中
                Y.append(label)

    # 将X和Y转换为NumPy数组
    X = np.array(X)
    Y = np.array(Y)
    
    index = np.arange(len(X))  
    np.random.shuffle(index)  
    X = X[index]  
    Y = Y[index]

    return X,Y

In [2]:
import torch
from torch import nn
import numpy as np

# 文件夹路径
folder_path = '/Users/mf/Documents/资料/教学相关/神经网络与深度学习/实验课题库/UCMerced_LandUse/train'

# 读取训练集和测试集数据
[train_img, train_lb] = read_UC('../实验课题库/UCMerced_LandUse/train')
[test_img, test_lb] = read_UC('../实验课题库/UCMerced_LandUse/validation')
# 对标签进行热编码
one_hot_train_lb = np.eye(9)[train_lb]
one_hot_test_lb= np.eye(9)[test_lb]

# 打印查看数据集格式
print('训练集图像格式为:', train_img.shape, '训练集标签格式为:', train_lb.shape,'热编码训练集标签格式为:', one_hot_train_lb.shape)
print('测试集图像格式为:', test_img.shape, '测试集标签格式为:', test_lb.shape,'热编码测试集标签格式为:', one_hot_test_lb.shape)

训练集图像格式为: (720, 3, 64, 64) 训练集标签格式为: (720,) 热编码训练集标签格式为: (720, 9)
测试集图像格式为: (720, 3, 64, 64) 测试集标签格式为: (720,) 热编码测试集标签格式为: (720, 9)


## 查看GPU是否可用

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


## 使用部分vgg16卷积层——搭建自己的网络

In [4]:
import torchvision.models as models
models.vgg16(weights='IMAGENET1K_V1')

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.w1 =nn.Linear(32768,100)
        self.w2 =nn.Linear(100,9)
        self.BN3=nn.BatchNorm1d(100)
        self.relu=nn.ReLU()
        self.vgg16= models.vgg16(weights='IMAGENET1K_V1').features

    def forward(self, x):
        x = self.vgg16[0](x)
        x = self.vgg16[1](x)
        x = self.vgg16[2](x)
        x = self.vgg16[3](x)
        x = self.vgg16[4](x)
        x = self.vgg16[5](x)
        x = self.vgg16[6](x)
        x = self.vgg16[7](x)
        x = self.vgg16[8](x)
        x = self.vgg16[9](x)
        x = x.view(x.size(0), -1)
        x = self.w1(x)
        x = self.BN3(x)
        x = self.relu(x)
        x = self.w2 (x)       
        return x

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (w1): Linear(in_features=32768, out_features=100, bias=True)
  (w2): Linear(in_features=100, out_features=9, bias=True)
  (BN3): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (vgg16): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3

In [8]:



# Initialize the loss function
loss_fn = nn.CrossEntropyLoss().to(device)

learning_rate = 5e-3
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

batch_size = 20
epochs = 4
batch_num=int(train_img.shape[0]/batch_size)
size = len(train_img)

model.train()
for t in range(epochs):
    
    correct=0.
    train_mean_loss=0.

    for batch in range(batch_num):
        X=train_img[batch*batch_size:(batch+1)*batch_size,]
        y=one_hot_train_lb[batch*batch_size:(batch+1)*batch_size,:]

        X=torch.tensor(X, dtype=torch.float32).to(device)
        y=torch.tensor(y, dtype=torch.float32).to(device)
        
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).mean()#.item()
        train_mean_loss+= loss#.item()
        print(correct,train_mean_loss)
    train_mean_loss /= batch_num
    correct /= batch_num

    print(f" Epoch:{t}, loss: {train_mean_loss:>8f},  Accuracy: {(100*correct):>0.1f}%")

torch.save(model,"vgg4_linear.pth")

tensor(1., device='mps:0') tensor(0.1575, device='mps:0', grad_fn=<AddBackward0>)
tensor(2., device='mps:0') tensor(0.2767, device='mps:0', grad_fn=<AddBackward0>)
tensor(3., device='mps:0') tensor(0.4305, device='mps:0', grad_fn=<AddBackward0>)
tensor(4., device='mps:0') tensor(0.5447, device='mps:0', grad_fn=<AddBackward0>)
tensor(5., device='mps:0') tensor(0.6998, device='mps:0', grad_fn=<AddBackward0>)
tensor(6., device='mps:0') tensor(0.8459, device='mps:0', grad_fn=<AddBackward0>)
tensor(7., device='mps:0') tensor(0.9830, device='mps:0', grad_fn=<AddBackward0>)
tensor(8., device='mps:0') tensor(1.1043, device='mps:0', grad_fn=<AddBackward0>)
tensor(9., device='mps:0') tensor(1.2255, device='mps:0', grad_fn=<AddBackward0>)
tensor(10., device='mps:0') tensor(1.3526, device='mps:0', grad_fn=<AddBackward0>)
tensor(11., device='mps:0') tensor(1.4887, device='mps:0', grad_fn=<AddBackward0>)
tensor(12., device='mps:0') tensor(1.6155, device='mps:0', grad_fn=<AddBackward0>)
tensor(13., d

In [21]:
#torch.load("vgg4_linear.pth")
model.eval()
test_loss, correct = 0, 0
with torch.no_grad():
        X=torch.tensor(test_img, dtype=torch.float32).to(device)
        y=torch.tensor(one_hot_test_lb, dtype=torch.float32).to(device)
        pred = model(X)
        test_loss = np.mean(loss_fn(pred, y).item())
        correct = (pred.argmax(1) == y.argmax(1)).type(torch.float).mean().item()
print(f"Test Accuracy: {(100*correct):>0.1f}%, Test Avg loss: {test_loss:>8f} \n")

Test Accuracy: 99.9%, Test Avg loss: 0.242083 

